Model Training

In [63]:
import pandas as pd

# path = r"C:\Users\Ehsan\Documents\Disertation\Word - June to\stock-market\Part4\All Ficnal data.xlsx"
# data = pd.read_excel(path)
# path = 'https://raw.githubusercontent.com/ehsanh123/stock_market_prediction/main/Part4/Financial%20Repors/All%20Ficnal%20data.xlsx'

path = r'C:\Users\Ehsan\OneDrive\Documents\Disertation\Word - June to\stock-market\Part4\All Ficnal data.xlsx'

excel_file = pd.ExcelFile(path)

# excel_file = pd.ExcelFile(path)
Comapnies = excel_file.sheet_names

Mark1 = pd.read_excel(excel_file, sheet_name='Marks & Spencer')
JD  = pd.read_excel(excel_file, sheet_name='JD sports')
ASOS = pd.read_excel(excel_file, sheet_name='Associated British Foods')
Next = pd.read_excel(excel_file, sheet_name='Next')
Ocado = pd.read_excel(excel_file, sheet_name='Ocado')
Tesco = pd.read_excel(excel_file, sheet_name='Tesco')
Sainsbury = pd.read_excel(excel_file, sheet_name='SaintBury\'s')
king = pd.read_excel(excel_file, sheet_name='King Fisher')

Mark1['Company'] = 'Marks & Spencer'
JD['Company'] = 'JD sports'
ASOS['Company'] = 'Associated British Foods'
Next['Company'] = 'Next'
Ocado['Company'] = 'Ocado'
Tesco['Company'] = 'Tesco'
Sainsbury['Company'] = "SaintBury's"
king['Company'] = 'King Fisher'
Data = pd.concat([Mark1, JD, ASOS, Next, Ocado, Tesco, Sainsbury, king], ignore_index=True)
Data1 = Data.copy()

In [64]:
Columns = [#'Year', 
           'Revenue', 'Operating Profit', 'Operating Margin (%)',
       #'Profit Before Tax ', 
       'Profit After Tax', 'Profit Margin (%)',
       'Basic ESP', 'Total Equity', 'Net Cash', 'Debt-to-Equity(%)',
       'Interim Dividend', 'Free Cash Flow', 'Avg Close Price',
       'Net From Oprating', 'CapEx', 'CEO Change', #'Company'
       ]
Double_Columns = [#'Year', 
    'Revenue', 'Operating Profit', #'Operating Margin (%)',
    #'Profit Before Tax ', 
    'Profit After Tax', #'Profit Margin (%)',
    #'Basic ESP', #'Total Equity', 'Net Cash','Debt-to-Equity(%)',
    #'Interim Dividend', #'Free Cash Flow', 'Avg Close Price',
    'Net From Oprating', 'CapEx', #'Company'
    ]

for idx, row in Data.iterrows():
    y1 = row['Year']
    if y1 % 1 != 0:
            for col in Double_Columns:
                Data1.at[idx, col ] = row[col] * 2

In [65]:
PCT_Columns = [#'Year', 
                'Revenue', 'Operating Profit', 
        #    'Profit After Tax', 
       'Free Cash Flow', #'Avg Close Price',
       'Net From Oprating', 'CapEx', 
       ]
Drivit_Columns = [
                'Operating Margin (%)',
           'Profit Margin (%)',
           'Debt-to-Equity(%)',
]
for id in PCT_Columns:
    Data1[id+'_PCT'] = Data1[id].pct_change() * 100
    Data1[id+'_PCT'] = Data1[id+'_PCT'].clip(upper=150, lower=-150)
    Data1.loc[Data1['Year'] == 2015, id+'_PCT'] = 0
for id in Drivit_Columns:
    Data1[id+'_Dri'] = Data1[id].diff()

In [66]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
#
Colmn = [
    'Revenue',  'Operating Profit',
    #    'Profit After Tax', 
       'Basic ESP',   #  'Total Equity',
         'Net Cash',   #    'Interim Dividend', 
       'Free Cash Flow','Revenue_PCT',
       'Operating Profit_PCT',  #  'Profit After Tax_PCT', 
       'Free Cash Flow_PCT', 'Net From Oprating_PCT', 
       'CapEx_PCT', 'Operating Margin (%)_Dri',
       'Profit Margin (%)_Dri','Debt-to-Equity(%)_Dri'
       ]
# Prepare input and output
Data3 = Data1.copy()
# Data3['C_Y'] = Data3['Company'] + Data3['Year'].astype(str)
# Data3.set_index('C_Y', inplace=True)
Data3.set_index('Company', inplace=True)

X = Data3[Colmn]


y = Data3[['Avg Close Price' , 'Avg_FSTE_Price']]

df = pd.concat([X, y], axis=1)

# Drop rows with NaNs in X (and optionally in target)
df = df.dropna(subset=X.columns)  # and maybe also ['target']

X_clean = df[X.columns]
y_clean = df[['Avg Close Price' , 'Avg_FSTE_Price']]

In [67]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import numpy as np

X_train, X_test, y_train1, y_test1 = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)

Standard_Columns = [
    'Operating Profit', 
    'Profit After Tax', 
   'Basic ESP', 
   'Net Cash',
   'Interim Dividend', 
   'Free Cash Flow', 
   'Operating Margin (%)_Dri',
    'Profit Margin (%)_Dri', 
    'Debt-to-Equity(%)_Dri'
]
log_Columns = [#'Year',
    'Revenue',
    'Total Equity'
]
PCT_Columns= [#'Year', 
    'Revenue_PCT',
    'Operating Profit_PCT', 
    'Profit After Tax_PCT', 
    'Free Cash Flow_PCT',
    'Net From Oprating_PCT', 
    'CapEx_PCT'
]
# meta_train[['Company','year']] = X_train[['Company','year']]
# meta_test[['Company','year']] = X_test[['Company','year']]

X_train, X_test, y_train1, y_test1 = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)

# meta_train[['Company','year']] = X_train[['Company','year']]
# meta_test[['Company','year']] = X_test[['Company','year']]

scaler_standard = StandardScaler()
scaler_minmax_pct  = MinMaxScaler()
scaler_minmax_log = MinMaxScaler() # 
scaler_minmax_out = MinMaxScaler(feature_range=(-1, 1))

for col in Standard_Columns:
    if col not in Colmn: continue
    scaler_standard.fit(X_train[[col]])
    X_train[col] = scaler_standard.transform(X_train[[col]])
    X_train[col] = X_train[col].clip(upper=3, lower=-3) / 3

    X_test[col] = scaler_standard.transform(X_test[[col]])
    X_test[col] = X_test[col].clip(upper=3, lower=-3) / 3

for col in log_Columns:
    if col not in Colmn: continue

    X_test[col] = np.log1p(X_test[col])
    X_train[col] = np.log1p(X_train[col])

    scaler_minmax_log.fit(X_train[[col]])

    X_test[col] = scaler_minmax_log.transform(X_test[[col]])
    X_train[col] = scaler_minmax_log.transform(X_train[[col]])

for col in PCT_Columns:

    if col not in Colmn: continue

    scaler_minmax_pct.fit(X_train[[col]])

    X_train[col] = scaler_minmax_pct.transform(X_train[[col]])
    X_test[col] = scaler_minmax_pct.transform(X_test[[col]])



In [68]:
y_train2 = y_train1.copy()
y_test2 = y_test1.copy()
###################
def classify_price_change1(pct):
    if pct < 0:
        return 0
    #False
    else:
        return 1

out_cols = ['Avg Close Price_dir' ,'Avg_FSTE_Price_dir']

y_train2['dif_from_FSTE'] = y_train1['Avg Close Price'] - y_train1['Avg_FSTE_Price']
y_test2['dif_from_FSTE'] = y_test1['Avg Close Price'] - y_test1['Avg_FSTE_Price']

y_train3 = y_train2.diff()
y_test3 = y_test2.diff()

y_train = y_train3['dif_from_FSTE'].apply(classify_price_change1)
y_test = y_test3['dif_from_FSTE'].apply(classify_price_change1)

In [69]:
from sklearn.metrics import accuracy_score, classification_report

from sklearn.ensemble import RandomForestClassifier
# Train a simple classifier
clf1 = RandomForestClassifier(random_state=42, n_estimators= 70)
# Train the model
clf1.fit(X_train, y_train)

# Predict on test set
y_pred = clf1.predict(X_test)

# Evaluate performance
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Drop', 'Rise']))

Accuracy: 0.7941176470588235

Classification Report:
               precision    recall  f1-score   support

        Drop       0.78      0.82      0.80        17
        Rise       0.81      0.76      0.79        17

    accuracy                           0.79        34
   macro avg       0.80      0.79      0.79        34
weighted avg       0.80      0.79      0.79        34



Deplyment of Long Term Model

In [70]:
import os, io, tempfile
import numpy as np
import pandas as pd
import gradio as gr

# -------- Config --------
IN_COLS = [
    "Revenue",
    "Operating Profit",
    "Profit After Tax",     # optional
    "Basic EPS",            # or "Basic ESP"
    "Total Equity",         # optional (for D/E)
    "Total Debt",           # optional (for D/E)
    "Net Cash",             # optional
    "Free Cash Flow",
    "Net From Operating",   # your spelling later maps to "Net From Oprating"
    "CapEx",
]

OUT_COLS = [
    "Revenue",
    "Operating Profit",
    "Basic ESP",
    "Net Cash",
    "Free Cash Flow",
    "Revenue_PCT",
    "Operating Profit_PCT",
    "Free Cash Flow_PCT",
    "Net From Oprating_PCT",
    "CapEx_PCT",
    "Operating Margin (%)_Dri",
    "Profit Margin (%)_Dri",
    "Debt-to-Equity(%)_Dri",
]

ALIASES = {
    "basic eps": "Basic EPS",
    "basic esp": "Basic ESP",            # accept ESP spelling too
    "eps (basic)": "Basic EPS",
    "net from operating": "Net From Operating",
    "net from oprating": "Net From Operating",
    "operating profit": "Operating Profit",
    "profit after tax": "Profit After Tax",
    "revenue": "Revenue",
    "total equity": "Total Equity",
    "total debt": "Total Debt",
    "net cash": "Net Cash",
    "free cash flow": "Free Cash Flow",
    "capex": "CapEx",
}


def _norm_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize column names using ALIASES; keep unknowns as-is."""
    rename = {}
    for c in df.columns:
        key = str(c).strip().lower()
        rename[c] = ALIASES.get(key, c)
    return df.rename(columns=rename)

def _to_float_series(df: pd.DataFrame, col: str) -> pd.Series:
    if col not in df.columns:
        return pd.Series(dtype=float)
    s = pd.to_numeric(df[col], errors="coerce")
    return s

def _pct(curr: float, prev: float) -> float:
    if pd.isna(curr) or pd.isna(prev) or prev == 0:
        return np.nan
    return 100.0 * (curr - prev) / abs(prev)


In [71]:

def compute_features(prev_df: pd.DataFrame, curr_df: pd.DataFrame) -> pd.DataFrame:
    """Compute requested outputs for *each row* (supports multiple rows). Rows align by index order."""
    # Normalize column names
    prev_df = _norm_cols(prev_df.copy())
    curr_df = _norm_cols(curr_df.copy())

    # Ensure both frames have the same number of rows; align by index
    n = max(len(prev_df), len(curr_df))
    prev_df = prev_df.reindex(range(n))
    curr_df = curr_df.reindex(range(n))

    rows = []
    for i in range(n):
        prev_row = prev_df.iloc[i] if i < len(prev_df) else pd.Series(dtype=float)
        curr_row = curr_df.iloc[i] if i < len(curr_df) else pd.Series(dtype=float)

        # pull values (float)
        rev_prev = pd.to_numeric(prev_row.get("Revenue"), errors="coerce")
        rev_curr = pd.to_numeric(curr_row.get("Revenue"), errors="coerce")

        op_prev  = pd.to_numeric(prev_row.get("Operating Profit"), errors="coerce")
        op_curr  = pd.to_numeric(curr_row.get("Operating Profit"), errors="coerce")

        fcf_prev = pd.to_numeric(prev_row.get("Free Cash Flow"), errors="coerce")
        fcf_curr = pd.to_numeric(curr_row.get("Free Cash Flow"), errors="coerce")

        nfo_prev = pd.to_numeric(prev_row.get("Net From Operating"), errors="coerce")
        nfo_curr = pd.to_numeric(curr_row.get("Net From Operating"), errors="coerce")

        capex_prev = pd.to_numeric(prev_row.get("CapEx"), errors="coerce")
        capex_curr = pd.to_numeric(curr_row.get("CapEx"), errors="coerce")

        pat_curr = pd.to_numeric(curr_row.get("Profit After Tax"), errors="coerce")
        pat_prev = pd.to_numeric(prev_row.get("Profit After Tax"), errors="coerce")

        total_equity_curr = pd.to_numeric(curr_row.get("Total Equity"), errors="coerce")
        total_equity_prev = pd.to_numeric(prev_row.get("Total Equity"), errors="coerce")

        total_debt_curr   = pd.to_numeric(curr_row.get("Total Debt"), errors="coerce")
        total_debt_prev   = pd.to_numeric(prev_row.get("Total Debt"), errors="coerce")

        net_cash = pd.to_numeric(curr_row.get("Net Cash"), errors="coerce")

        # EPS / ESP mapping
        basic_eps = pd.to_numeric(
            curr_row.get("Basic EPS", curr_row.get("Basic ESP", np.nan)),
            errors="coerce"
        )
        # Output wants "Basic ESP" — keep same numeric value
        basic_esp_out = basic_eps

        # % changes vs previous report
        rev_pct  = _pct(rev_curr, rev_prev)
        op_pct   = _pct(op_curr, op_prev)
        fcf_pct  = _pct(fcf_curr, fcf_prev)
        nfo_pct  = _pct(nfo_curr, nfo_prev)
        capex_pct = _pct(capex_curr, capex_prev)

        
        ##################
        # print(f' pat_curr: {pat_curr} , rev_curr {rev_curr}')
        # print(f' rev_curr: {rev_curr} , rev_prev {rev_prev}')
        profit_mrgin_curt = (pat_curr / rev_curr) *100
        profit_mrgin_prv = (pat_prev / rev_prev) *100
        # print(f' profit_mrgin_curt: {profit_mrgin_curt} , profit_mrgin_prv {profit_mrgin_prv}')
        prof_margin = np.nan
        prof_margin = profit_mrgin_curt - profit_mrgin_prv
        # print(f' profit_mrgin_drv: {prof_margin} ')
        ###########
        # print(' op_curr ', op_curr, ' rev_curr ', rev_curr)
        # print(' op_prev ', op_prev, ' rev_prev ', rev_prev)
        opration_margin_curt = (op_curr / rev_curr  ) *100
        opration_margin_prv =  ( op_prev / rev_prev ) *100

        # print(f' opration_margin_curt: {opration_margin_curt} , opration_margin_prv {opration_margin_prv}')

        op_margin = np.nan
        op_margin = opration_margin_curt - opration_margin_prv
        # print(f' opration_margin_drv: {op_margin} ')
        #############
        # print(f' total_debt_curr: {total_debt_curr} , total_equity_curr {total_equity_curr}')
        
        debt_equity_curr = (total_debt_curr / total_equity_curr ) *100
        debt_equity_prev = (total_debt_prev / total_equity_prev ) *100

        # print(f' debt/equity curr: {debt_equity_curr} , old {debt_equity_prev}')

        de_ratio = np.nan
        # if pd.notna(total_debt) and pd.notna(total_equity) and total_equity != 0:
        de_ratio = debt_equity_curr - debt_equity_prev
        # print(f' debt/equity drivite: {de_ratio}')

        row = {
            "Revenue": rev_curr,
            "Operating Profit": op_curr,
            "Basic ESP": basic_esp_out,
            "Net Cash": net_cash,
            "Free Cash Flow": fcf_curr,
            "Revenue_PCT": rev_pct,
            "Operating Profit_PCT": op_pct,
            "Free Cash Flow_PCT": fcf_pct,
            "Net From Oprating_PCT": nfo_pct,      # keep your requested spelling
            "CapEx_PCT": capex_pct,
            "Operating Margin (%)_Dri": op_margin,
            "Profit Margin (%)_Dri":prof_margin,
            "Debt-to-Equity(%)_Dri": de_ratio,
        }
        rows.append(row)

    out = pd.DataFrame(rows, columns=OUT_COLS)
    return out

def _csv_from_df(df: pd.DataFrame) -> str:
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".csv")
    df.to_csv(tmp.name, index=False)
    return tmp.name

def compute_from_manual(prev_df: pd.DataFrame, curr_df: pd.DataFrame):
    if prev_df is None or curr_df is None or (len(curr_df) == 0):
        return None, None
    res = compute_features(prev_df, curr_df)
    # csv_path = _csv_from_df(res)
    return res

def compute_from_csv(prev_file, curr_file):
    if prev_file is None or curr_file is None:
        return None, None
    prev_df = pd.read_csv(prev_file.name)
    curr_df = pd.read_csv(curr_file.name)
    prev_df = prev_df.reindex(columns=IN_COLS, fill_value=np.nan)  # keep order if present
    curr_df = curr_df.reindex(columns=IN_COLS, fill_value=np.nan)
    res = compute_features(prev_df, curr_df)
    csv_path = _csv_from_df(res)
    return res, csv_path


In [82]:
# Use your full header set so it shows exactly what you want original values

prev_row = {
    "Revenue": 15962,
    "Operating Profit": 1057.2,
    "Profit After Tax": 1575.2 ,
    "Basic EPS": 14.6,
    "Total Equity": 3031.0,
    "Net Cash": 618.7 ,
    # "Debt-to-Equity(%)": 20.41,
    "Total Debt": 618.62 ,
    "Free Cash Flow": 1254.9,
    "Net From Operating": 395.8,
    "CapEx": 720.2,
}
       
curr_row = {
    "Revenue": 13816.8,
    "Operating Profit": 624.3,
    "Profit After Tax": 291.9,
    "Basic EPS": 0.1,
    "Total Equity": 2951.0,
    "Net Cash": 864.5 ,
    # "Debt-to-Equity(%)": 29.3,
    "Total Debt" : 864.64 ,
    "Free Cash Flow": 16.9,
    "Net From Operating": 13,
    "CapEx": 696.1
}

In [73]:
# --- Define the Prediction Function for Gradio (takes original values) ---
def predict_stock_trend(*features_raw):
    input_df = pd.DataFrame([features_raw], columns=X.columns)

    # Standard Scaling
    for col in Standard_Columns:
        if col not in Colmn: continue
        input_df[col] = scaler_standard.transform(input_df[[col]])
        input_df[col] = input_df[col].clip(upper=3, lower=-3) / 3

    for col in log_Columns:
        if col not in Colmn: continue
        input_df[col] = np.log1p(input_df[col])
        input_df[col] = scaler_minmax_log.transform(input_df[[col]])
        
    for col in PCT_Columns:
        if col not in Colmn: continue
        input_df[col] = scaler_minmax_pct.transform(input_df[[col]])

    # Make prediction using the normalized input
    prediction = clf1.predict(input_df)[0]
    probabilities = clf1.predict_proba(input_df)[0] # Get probabilities for the single input


    # Map numerical prediction to 'Drop' or 'Rise'
     # Map numerical prediction to 'Drop' or 'Rise' and get the corresponding probability
    if prediction == 0:
        predicted_label = "Drop"
        confidence = probabilities[0] # Probability of 'Drop'
    else:
        predicted_label = "Rise"
        confidence = probabilities[1]
    
    return predicted_label, f"{confidence:.4f}" 
    # Format probability to 4 decimal places


In [84]:
# ---- Helper: pull first row from computed table into the RF inputs order ----
import pandas as pd
def _pull_first_row_for_rf(df: pd.DataFrame):
    if df is None or len(df) == 0:
        return [None] * len(Colmn)
    row = df.iloc[0]
    vals = []
    for c in Colmn:
        v = row[c] if c in row.index else np.nan
        # coerce to float if possible
        try:
            # v = float(pd.to_numeric(v, errors="coerce"))
            v = round(float(pd.to_numeric(v, errors="coerce")), 2)
        except Exception:
            v = None
        vals.append(v)
    return vals

In [ ]:
with gr.Blocks(title="Financial Feature Builder") as demo:
    gr.Markdown("## Stocks Trend from Financial Features (Current vs Previous Reports)")

    with gr.Tabs():
        with gr.Tab("Manual Entry"):
            gr.Markdown("Pre-filled with Tesco (2024.5) * 2 and (2025) report values. Edit if needed.")

            with gr.Row():
                # prev_df = [gr.Number(label=Colmn[i]) for i in range(num_features)]

                prev_df = gr.Dataframe(
                    headers=IN_COLS,
                    value=pd.DataFrame([prev_row], columns=IN_COLS),
                    type="pandas",
                    wrap=True,
                    label="Previous Report"
                )
            with gr.Row():
                curr_df = gr.Dataframe(
                    headers=IN_COLS,
                    value=pd.DataFrame([curr_row], columns=IN_COLS),
                    type="pandas",
                    wrap=True,
                    label="Current Report"
                )

            btn_manual = gr.Button("Compute Features")
            out_table_m = gr.Dataframe(label="Computed Features", interactive=False)
            # out_csv_m   = gr.File(label="Download CSV")

            btn_manual.click(
                fn=compute_from_manual,        # <- keep your existing function
                inputs=[prev_df, curr_df],
                outputs=[out_table_m]
            )

    with gr.Row():
        # 3) UI controls: load from table + predict
        load_btn = gr.Button("Load inputs from Computed Features ↑")
        num_features = len(Colmn)
        input_components = [gr.Number(label=Colmn[i]) for i in range(num_features)]



    # when clicked, copy table → inputs (maps list → each Number component)
    load_btn.click(
        fn=_pull_first_row_for_rf,
        inputs=[out_table_m],
        outputs=input_components
    )
        # (keep your "Upload CSVs" tab as-is)
    with gr.Row():
        pred_label = gr.Textbox(label="Predicted Stock Trend")
        pred_prob  = gr.Textbox(label="Prediction Probability")
        btn_pred = gr.Button("Prdict Stock Trend")
    # btn_pred.click(
    #     fn=predict_stock_trend,
    #     inputs=out_table_m,
    #     outputs=[pred_label, pred_prob]
    # )
    btn_pred.click(
        fn=lambda *vals: predict_stock_trend(*vals),
        inputs=input_components,
        outputs=[pred_label, pred_prob]
    )
demo.launch()


* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


c:\Users\Ehsan\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature names unseen at fit time:
- Operating Profit
Feature names seen at fit time, yet now missing:
- Debt-to-Equity(%)_Dri

  warnings.warn(message, FutureWarning)
c:\Users\Ehsan\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature names unseen at fit time:
- Basic ESP
Feature names seen at fit time, yet now missing:
- Debt-to-Equity(%)_Dri

  warnings.warn(message, FutureWarning)
c:\Users\Ehsan\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature